# Exercise 03: Bronze Layer Ingestion Pipeline

Build a complete Bronze layer ingestion pipeline.

## Objectives
- Implement partitioned data ingestion
- Add proper metadata tracking
- Handle multiple file types
- Build reusable ingestion functions

In [ ]:
import boto3
import pandas as pd
from datetime import datetime
import os
import json

s3 = boto3.client('s3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

BRONZE_BUCKET = 'bronze'

# Ensure bucket exists
try:
    s3.head_bucket(Bucket=BRONZE_BUCKET)
except:
    s3.create_bucket(Bucket=BRONZE_BUCKET)

print("Ready!")

## Task 1: Create Sample Data Files

Create sample sales and customer data files.

In [ ]:
# Create sample sales data
sales_df = pd.DataFrame({
    'transaction_id': range(1001, 1011),
    'customer_id': [101, 102, 101, 103, 104, 102, 105, 101, 103, 106],
    'product_id': [1, 2, 3, 1, 4, 5, 2, 3, 4, 1],
    'quantity': [1, 2, 1, 3, 1, 2, 1, 2, 1, 1],
    'amount': [999.99, 59.98, 79.99, 2999.97, 299.99, 299.98, 29.99, 159.98, 299.99, 999.99],
    'transaction_date': ['2024-01-15'] * 10
})
sales_df.to_csv('/tmp/sales_20240115.csv', index=False)

# Create sample customer data (JSON)
customers = [
    {'id': 101, 'name': 'Alice Smith', 'email': 'alice@example.com', 'tier': 'gold'},
    {'id': 102, 'name': 'Bob Jones', 'email': 'bob@example.com', 'tier': 'silver'},
    {'id': 103, 'name': 'Carol White', 'email': 'carol@example.com', 'tier': 'bronze'},
    {'id': 104, 'name': 'David Brown', 'email': 'david@example.com', 'tier': 'silver'},
    {'id': 105, 'name': 'Eve Davis', 'email': 'eve@example.com', 'tier': 'gold'},
    {'id': 106, 'name': 'Frank Miller', 'email': 'frank@example.com', 'tier': 'bronze'}
]
with open('/tmp/customers_20240115.json', 'w') as f:
    json.dump(customers, f)

print("Sample files created!")
print(f"Sales: {len(sales_df)} records")
print(f"Customers: {len(customers)} records")

## Task 2: Build Key Generator

Create a function that generates partitioned keys.

In [ ]:
def generate_bronze_key(source_name, filename, timestamp=None):
    """
    Generate a partitioned key for bronze layer.
    
    Format: {source}/year={YYYY}/month={MM}/day={DD}/{filename}_{HHMMSS}.{ext}
    
    Args:
        source_name: Name of the data source (e.g., 'sales', 'customers')
        filename: Original filename
        timestamp: Optional datetime, defaults to now
    
    Returns:
        Partitioned key string
    """
    # YOUR CODE HERE
    pass

# Test
test_key = generate_bronze_key('sales', 'transactions.csv')
print(f"Generated key: {test_key}")
# Expected format: sales/year=2024/month=01/day=17/transactions_143022.csv

## Task 3: Build Ingestion Function

Create a function to ingest files with metadata.

In [ ]:
def ingest_to_bronze(local_path, source_name, extra_metadata=None):
    """
    Ingest a file to the bronze layer with metadata.
    
    Args:
        local_path: Path to local file
        source_name: Name of the data source
        extra_metadata: Optional dict of additional metadata
    
    Returns:
        The S3 key where file was uploaded
    """
    # YOUR CODE HERE
    # 1. Get filename from path
    # 2. Generate partitioned key
    # 3. Build metadata dict (source, original-filename, ingestion-time, file-size)
    # 4. Upload with metadata
    # 5. Return the key
    pass

# Test
key = ingest_to_bronze('/tmp/sales_20240115.csv', 'sales')
print(f"Ingested to: {key}")

## Task 4: Ingest All Sample Data

Ingest both sales and customer files.

In [ ]:
# Ingest sales data
sales_key = ingest_to_bronze(
    '/tmp/sales_20240115.csv', 
    'sales',
    {'record-count': '10', 'data-date': '2024-01-15'}
)

# Ingest customer data
customers_key = ingest_to_bronze(
    '/tmp/customers_20240115.json',
    'customers',
    {'record-count': '6', 'data-date': '2024-01-15'}
)

print(f"\nIngested files:")
print(f"  Sales: {sales_key}")
print(f"  Customers: {customers_key}")

## Task 5: List Ingested Data

Create a function to list all ingested data for a source.

In [ ]:
def list_bronze_data(source_name=None):
    """
    List all data in bronze layer, optionally filtered by source.
    
    Args:
        source_name: Optional source filter
    
    Returns:
        List of object info dicts
    """
    # YOUR CODE HERE
    pass

# List all bronze data
print("All Bronze Data:")
for item in list_bronze_data():
    print(f"  {item}")

print("\nSales Data Only:")
for item in list_bronze_data('sales'):
    print(f"  {item}")

## Task 6: Verify Metadata

Retrieve and display metadata for ingested files.

In [ ]:
def get_object_metadata(key):
    """Get metadata for an object"""
    # YOUR CODE HERE
    pass

# Check metadata
print("Sales file metadata:")
print(get_object_metadata(sales_key))

print("\nCustomers file metadata:")
print(get_object_metadata(customers_key))

## Task 7: Build Complete Ingestion Class

Combine everything into a reusable class.

In [ ]:
class BronzeIngestion:
    """Bronze layer ingestion pipeline"""
    
    def __init__(self, endpoint='http://minio:9000', 
                 access_key='minioadmin', secret_key='minioadmin',
                 bucket='bronze'):
        self.s3 = boto3.client('s3',
            endpoint_url=endpoint,
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key
        )
        self.bucket = bucket
        self._ensure_bucket()
    
    def _ensure_bucket(self):
        # YOUR CODE: Create bucket if not exists
        pass
    
    def _generate_key(self, source, filename):
        # YOUR CODE: Generate partitioned key
        pass
    
    def ingest_file(self, local_path, source, metadata=None):
        # YOUR CODE: Ingest file with metadata
        pass
    
    def list_data(self, source=None):
        # YOUR CODE: List ingested data
        pass
    
    def get_metadata(self, key):
        # YOUR CODE: Get object metadata
        pass

In [ ]:
# Test the class
ingestion = BronzeIngestion()

# Ingest files
key1 = ingestion.ingest_file('/tmp/sales_20240115.csv', 'sales')
key2 = ingestion.ingest_file('/tmp/customers_20240115.json', 'customers')

# List data
print("\nAll ingested data:")
for item in ingestion.list_data():
    print(f"  {item}")

## Congratulations! 🎉

You've built a complete Bronze layer ingestion pipeline. This is the foundation for all data processing in a modern data lake.

**Next**: Module 02 - PySpark Processing (Silver Layer)